[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/01_llava_architecture.ipynb)

# 01. LLaVA Architecture: Connecting Vision to LLMs

**LLaVA (Large Language and Vision Assistant)** is the blueprint for modern multimodal LLMs.

**This notebook covers:**
- LLaVA architecture — step-by-step diagram
- The projection layer: how images become "language"
- Two-stage training: pretrain projector → finetune end-to-end
- Build a mini-LLaVA from scratch
- How GPT-4V, Gemini, etc. extend this idea

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/05_Advanced_Topics")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

In [ ]:
# LLaVA Architecture Diagram

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('LLaVA Architecture', fontsize=20, fontweight='bold', pad=20)

# Image path
draw_architecture_block(ax, 3, 9, 3.5, 0.7, 'Input Image', '#E74C3C')
draw_architecture_block(ax, 3, 7.5, 3.5, 0.9, 'Vision Encoder\n(CLIP ViT-L, frozen)', '#E74C3C')
draw_architecture_block(ax, 3, 5.8, 3.5, 0.7, 'Image Features\n[N, 1024]', '#C0392B')
draw_architecture_block(ax, 3, 4.3, 3.5, 0.9, 'Projection (MLP)\nLinear→GELU→Linear\n[N, 1024] → [N, 4096]', '#F39C12')
ax.text(3, 3.3, 'Image "tokens"\n(same dim as text!)', ha='center', fontsize=10,
        color='#F39C12', fontweight='bold')

for y1, y2 in [(8.6, 8.0), (7.0, 6.2), (5.4, 4.8)]:
    draw_arrow(ax, (3, y1), (3, y2))

# Text path
draw_architecture_block(ax, 10, 9, 4, 0.7, 'User: "Describe this image"', '#3498DB')
draw_architecture_block(ax, 10, 7.5, 4, 0.7, 'Tokenize + Embed', '#3498DB')

draw_arrow(ax, (10, 8.6), (10, 8.0))

# Concatenation
draw_architecture_block(ax, 8, 5.5, 10, 0.9, 'Concatenate: [image_tokens, text_tokens]\n= [img1, img2, ..., imgN, user, describe, this, image]', '#9B59B6', fontsize=9)

draw_arrow(ax, (3, 3.8), (4.5, 6.0))
draw_arrow(ax, (10, 7.0), (10, 6.0))

# LLM
draw_architecture_block(ax, 8, 3.5, 10, 1.2, 'LLM Decoder (LLaMA, frozen or LoRA)\nCausal self-attention over all tokens', '#2ECC71', fontsize=11)
draw_architecture_block(ax, 8, 1.5, 6, 0.8, 'Generated Answer:\n"A cat sitting on a red sofa"', '#34495E')

draw_arrow(ax, (8, 4.9), (8, 4.2))
draw_arrow(ax, (8, 2.8), (8, 2.0))

# Annotations
ax.text(14.5, 7.5, 'Frozen', fontsize=10, color='#E74C3C', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.8))
ax.text(14.5, 4.3, 'Trainable\n(Stage 1)', fontsize=10, color='#F39C12', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FEF9E7', alpha=0.8))
ax.text(14.5, 3.5, 'LoRA\n(Stage 2)', fontsize=10, color='#2ECC71', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#D5F5E3', alpha=0.8))

plt.tight_layout()
plt.savefig('../assets/llava_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

## The Key Insight: Images as Language Tokens

LLaVA's trick is simple but brilliant:
1. Use a frozen CLIP vision encoder to get image features
2. Use an MLP to project image features into the **same dimension as text tokens**
3. Concatenate image tokens + text tokens
4. Feed everything into the LLM — it treats image tokens just like word tokens!

In [ ]:
# Build Mini-LLaVA from scratch

class VisionProjector(nn.Module):
    """Projects image features to LLM dimension (the trainable bridge)."""
    def __init__(self, vision_dim=512, llm_dim=256):
        super().__init__()
        self.projector = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )

    def forward(self, image_features):
        return self.projector(image_features)


class MiniVisionEncoder(nn.Module):
    """Small ViT as stand-in for CLIP vision encoder."""
    def __init__(self, img_size=32, patch_size=4, dim=128, n_layers=2):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, dim, patch_size, patch_size)
        self.pos = nn.Parameter(torch.randn(1, n_patches, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=4, dim_feedforward=dim*4, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2) + self.pos
        return self.norm(self.enc(x))  # [B, N_patches, dim]


class MiniLLM(nn.Module):
    """Small causal language model."""
    def __init__(self, vocab_size=500, dim=256, n_layers=3, max_len=128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_len, dim)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=4, dim_feedforward=dim*4, batch_first=True)
        self.decoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Linear(dim, vocab_size)
        self.dim = dim

    def forward(self, token_embeds):
        """Takes pre-embedded tokens (image + text already concatenated)."""
        B, T, D = token_embeds.shape
        pos = self.pos_emb(torch.arange(T, device=token_embeds.device)).unsqueeze(0)
        x = token_embeds + pos
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        x = self.decoder(x, mask=mask)
        return self.head(x)


class MiniLLaVA(nn.Module):
    """Complete LLaVA-style model."""
    def __init__(self, vocab_size=500, vision_dim=128, llm_dim=256):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(dim=vision_dim)
        self.projector = VisionProjector(vision_dim, llm_dim)
        self.llm = MiniLLM(vocab_size=vocab_size, dim=llm_dim)

    def forward(self, images, text_ids):
        # Get image features and project
        img_features = self.vision_encoder(images)       # [B, N_patches, vision_dim]
        img_tokens = self.projector(img_features)         # [B, N_patches, llm_dim]

        # Get text embeddings
        txt_tokens = self.llm.tok_emb(text_ids)           # [B, T, llm_dim]

        # Concatenate: [image_tokens, text_tokens]
        combined = torch.cat([img_tokens, txt_tokens], dim=1)  # [B, N+T, llm_dim]

        # Run through LLM
        logits = self.llm(combined)                       # [B, N+T, vocab_size]
        return logits


model = MiniLLaVA(vocab_size=500, vision_dim=128, llm_dim=256)
count_parameters(model)

imgs = torch.randn(2, 3, 32, 32)
txt = torch.randint(0, 500, (2, 10))
out = model(imgs, txt)
print(f"\nInput: images {imgs.shape}, text {txt.shape}")
print(f"Output logits: {out.shape}")
print(f"  First {64} tokens = image, last {10} = text predictions")

In [ ]:
# Visualize the two-stage training process

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('LLaVA Two-Stage Training', fontsize=18, fontweight='bold')

# Stage 1
ax = axes[0]
ax.set_xlim(0, 8); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('Stage 1: Pretrain Projector\n(595K image-text pairs, 1 epoch)', 
             fontsize=12, fontweight='bold', color='#F39C12')

draw_architecture_block(ax, 4, 7, 5, 0.7, 'Vision Encoder (FROZEN)', '#95A5A6')
draw_architecture_block(ax, 4, 5.5, 5, 0.9, 'Projection MLP (TRAINABLE)', '#F39C12')
draw_architecture_block(ax, 4, 3.8, 5, 0.7, 'LLM (FROZEN)', '#95A5A6')
draw_architecture_block(ax, 4, 2, 5, 0.7, 'Generate caption', '#2ECC71')

for y1, y2 in [(6.6, 6.0), (5.0, 4.2), (3.4, 2.5)]:
    draw_arrow(ax, (4, y1), (4, y2))

ax.text(4, 0.8, 'Goal: Align image features\nwith LLM embedding space',
        ha='center', fontsize=10, style='italic')

# Stage 2
ax = axes[1]
ax.set_xlim(0, 8); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('Stage 2: Instruction Tuning\n(150K instruction data, 1 epoch)',
             fontsize=12, fontweight='bold', color='#2ECC71')

draw_architecture_block(ax, 4, 7, 5, 0.7, 'Vision Encoder (FROZEN)', '#95A5A6')
draw_architecture_block(ax, 4, 5.5, 5, 0.9, 'Projection MLP (TRAINABLE)', '#F39C12')
draw_architecture_block(ax, 4, 3.8, 5, 0.7, 'LLM + LoRA (TRAINABLE)', '#2ECC71')
draw_architecture_block(ax, 4, 2, 5, 0.7, 'Follow instructions', '#2ECC71')

for y1, y2 in [(6.6, 6.0), (5.0, 4.2), (3.4, 2.5)]:
    draw_arrow(ax, (4, y1), (4, y2))

ax.text(4, 0.8, 'Goal: Learn to follow\nvisual instructions (Q&A, describe, etc.)',
        ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.savefig('../assets/llava_training_stages.png', dpi=150, bbox_inches='tight')
plt.show()

## Modern Multimodal LLM Comparison

| Model | Vision Encoder | LLM | Projection | Training |
|-------|---------------|-----|------------|----------|
| **LLaVA** | CLIP ViT-L | LLaMA-7B | 2-layer MLP | 2 stages |
| **LLaVA-1.5** | CLIP ViT-L/14 | Vicuna-13B | 2-layer MLP | 2 stages |
| **BLIP-2** | ViT-G | FlanT5/OPT | Q-Former | 3 stages |
| **InstructBLIP** | ViT-G | Vicuna | Q-Former | Instruction tuned |
| **GPT-4V** | Unknown | GPT-4 | Unknown | End-to-end |
| **Gemini** | Built-in | Built-in | Native | End-to-end |

**For low compute: LLaVA + QLoRA is the most accessible path.**

---
**Next:** `02_multimodal_beyond_vision.ipynb` - Audio, video, and beyond